## Logistic Regression via PyTorch


We load a lot of libraries.  

In [ ]:
import torch
# Plotting libraries
import bokeh
from bokeh.plotting import figure, output_notebook, show
from bokeh.models import Label
# numpy and pandas
import numpy as np
import pandas as pd
# tqdm makes progress bars
from tqdm.auto import tqdm
# we use train test split
from sklearn.model_selection import train_test_split

print(f"""Using Torch version {torch.__version__}.  
        CUDA is {'available' if torch.cuda.is_available() else 'not available'}. 
        MPS is {'available' if torch.backends.mps.is_available() else 'not available'}""")
gpu = 'mps' if torch.backends.mps.is_available() else 'cuda'
cpu = 'cpu'


print(f"Using bokeh version {bokeh.__version__}.")


print(f"Using pandas version {pd.__version__}.")




In [ ]:
# Set up plotting to notebook; use cpu for computations
output_notebook()
device = cpu

Logistic regression (for 15 features and 1 output) is a very simple neural network.

In [ ]:
class LogR(torch.nn.Module):
    """A simple 15x1 logistic regression model."""
    def __init__(self):
        super().__init__()
        self.logistic=torch.nn.Linear(15,1,dtype=torch.float32)

    def forward(self,x):
        return torch.sigmoid(self.logistic(x))

We load the "food" data that we used in our earlier look at logistic regression

In [ ]:
df = pd.read_csv('ifood_df.csv',delimiter=',')
features = df.columns[[0,4,5,6,7,8,9,10,11,12,13,14,24,36,37]]
df[features] = (df[features]-df[features].mean())/df[features].std()
x_train, x_test, y_train, y_test = train_test_split(df[features].values, df['Response'].values)



Create the model on the device; show the initial (randomly chosen) parameters.

In [ ]:
model = LogR().to(device)
print(model)
for name,param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Set up the training data as torch tensors.  We use binary cross entropy for the loss, and stochastic gradient descent for the optimization algorithm.

In [ ]:
Xt = torch.tensor(x_train,dtype=torch.float32,device=device)
Yt = torch.tensor(y_train,dtype=torch.float32,device=device)
criterion = torch.nn.functional.binary_cross_entropy
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)


In [ ]:
def train(model, criterion, optimizer, Xt, Yt):
    """One step through the training loop"""
    # reset the gradient calculations
    optimizer.zero_grad()

    # forward pass
    predicted = model(Xt)
    
    # compute the loss
    loss = criterion(torch.squeeze(predicted),Yt)
    
 

    # compute the gradients by backward propogation
    loss.backward()        
        
    # adjust the weights
    optimizer.step()
    
    return loss.item()

In [ ]:
def training_loop(model, data, target, learning_rate=.0001,threshold=1e-6,max_iter=100000):
    """Run the training loop and return the losses"""
    criterion = torch.nn.functional.binary_cross_entropy
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

    losses = []
    prior_loss=1000000
    for i in tqdm(range(max_iter)):
        loss = train(model,criterion,optimizer,Xt,Yt)
        losses.append(loss)
        if abs(loss-prior_loss) < threshold:
            break
        prior_loss = loss
        
    with torch.no_grad():
        prediction = model(Xt).round()
        print(f"Accuracy on training data is {((prediction - Yt.reshape(-1,1))==0).sum()/Xt.shape[0]}")
    
    return losses
    

In [ ]:
def plot_loss(losses):
    """Run the model and collect the losses; return a figure"""
    
    f=figure(title=f"Loss over time",x_axis_label="Epoch",y_axis_label="Loss")
    f.line(x=list(range(len(losses))),y=losses)

    
    return f

In [ ]:
model = LogR().to(device)
losses = training_loop(model, Xt, Yt)
show(plot_loss(losses))

Now we can check the accuracy on the test data

In [ ]:
Xtst = torch.tensor(x_test,dtype=torch.float32,device=device,requires_grad=False)
Ytst = torch.tensor(y_test, dtype = torch.float32, device=device, requires_grad=False)

In [ ]:
prediction = model(Xtst).round()
print(f"Accuracy is {((prediction - Ytst.reshape(-1,1))==0).sum()/Xtst.shape[0]}")
    